# HC18 nnU-Net 100-Epoch Parallel Training (Folds 1 & 2)
- Dataset: `Dataset501_HC18`
- Parallel training on 2x T4 GPUs:
  - Fold 1 on `cuda:0`
  - Fold 2 on `cuda:1`
- Trainer: `nnUNetTrainer_100epochs`
- Output: validation masks and `checkpoint_final.pth` backup for both folds.
- DO NOT PUSH before quota reset on Saturday 2026-11-21 06:00 Dhaka.


In [ ]:
!nvidia-smi
!pip install --upgrade pip
!pip install --quiet nnunetv2


In [ ]:
import os
import shutil
import json
from pathlib import Path

os.environ["nnUNet_raw"] = "/kaggle/working/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/kaggle/working/nnUNet_results"

raw_dir = Path("/kaggle/working/nnUNet_raw")
prep_dir = Path("/kaggle/working/nnUNet_preprocessed")
res_dir = Path("/kaggle/working/nnUNet_results")
out_dir = Path("/kaggle/working/out")

for d in [raw_dir, prep_dir, res_dir, out_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Locate dataset.json
dataset_dir = raw_dir / "Dataset501_HC18"
dataset_jsons = list(Path("/kaggle/input").rglob("dataset.json"))
assert dataset_jsons, "Could not find dataset.json in /kaggle/input"
src_dir = dataset_jsons[0].parent
shutil.copytree(src_dir, dataset_dir, dirs_exist_ok=True)
print(f"Dataset unpacked to {dataset_dir}")

assert (dataset_dir / "dataset.json").is_file()
assert (dataset_dir / "imagesTr").is_dir()
assert (dataset_dir / "labelsTr").is_dir()
print(f"Images: {len(list((dataset_dir / 'imagesTr').glob('*.png')))}, Labels: {len(list((dataset_dir / 'labelsTr').glob('*.png')))}")


In [ ]:
!nnUNetv2_plan_and_preprocess -d 501 -c 2d --verify_dataset_integrity


In [ ]:
import json
import shutil
from pathlib import Path

src_split = dataset_dir / "splits_final.json"
dst_split = prep_dir / "Dataset501_HC18" / "splits_final.json"
dst_split.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src_split, dst_split)

with open(dst_split, "r") as f:
    splits = json.load(f)

print(f"Splits verified: {len(splits)} folds.")
for fold_num in [1, 2]:
    n_tr = len(splits[fold_num]["train"])
    n_va = len(splits[fold_num]["val"])
    print(f"  Fold {fold_num}: {n_tr} train, {n_va} val")
    overlap = set(splits[fold_num]["train"]).intersection(set(splits[fold_num]["val"]))
    assert len(overlap) == 0, f"Leakage detected in fold {fold_num}"


In [ ]:
import subprocess
import time
import sys
from pathlib import Path

log_a = Path("/kaggle/working/train_fold_1.log")
log_b = Path("/kaggle/working/train_fold_2.log")

cmd_a = "CUDA_VISIBLE_DEVICES=0 nnUNetv2_train 501 2d 1 -tr nnUNetTrainer_100epochs"
cmd_b = "CUDA_VISIBLE_DEVICES=1 nnUNetv2_train 501 2d 2 -tr nnUNetTrainer_100epochs"

print("Starting parallel training on 2 GPUs:")
print("  GPU 0:", cmd_a)
print("  GPU 1:", cmd_b)

start_time = time.time()
with open(log_a, "w") as fa, open(log_b, "w") as fb:
    p_a = subprocess.Popen(cmd_a, shell=True, stdout=fa, stderr=subprocess.STDOUT)
    p_b = subprocess.Popen(cmd_b, shell=True, stdout=fb, stderr=subprocess.STDOUT)

    print("Processes launched. Monitoring log outputs for first 5 epochs...")
    reported_epochs = set()
    epoch_times = { 1: [], 2: [] }

    while p_a.poll() is None or p_b.poll() is None:
        time.sleep(10)
        for fold, lf in [(1, log_a), (2, log_b)]:
            if lf.exists():
                lines = lf.read_text(errors="ignore").splitlines()
                for l in lines[-15:]:
                    if "Epoch " in l and "time:" in l.lower():
                        ep_key = (fold, l.strip())
                        if ep_key not in reported_epochs:
                            reported_epochs.add(ep_key)
                            print(f"[Fold {fold}] {l.strip()}")
                            # Check if over 200 s
                            try:
                                t_str = l.lower().split("time:")[1].split("s")[0].strip()
                                t_sec = float(t_str)
                                epoch_times[fold].append(t_sec)
                                if t_sec > 200.0:
                                    print(f"WARNING: [Fold {fold}] Epoch time {t_sec:.1f} s exceeds 200 s!")
                            except Exception:
                                pass

        if len(epoch_times[1]) >= 5 and len(epoch_times[2]) >= 5:
            print("First 5 epochs completed on both folds. Continuing in background...")
            break

    # Wait for completion
    p_a.wait()
    p_b.wait()

elapsed = time.time() - start_time
print(f"Parallel training finished in {elapsed:.1f} s ({elapsed / 60:.1f} min)")
assert p_a.returncode == 0, f"Fold 1 failed with return code {p_a.returncode}"
assert p_b.returncode == 0, f"Fold 2 failed with return code {p_b.returncode}"


In [ ]:
import shutil
from pathlib import Path

trainer_base = res_dir / "Dataset501_HC18" / "nnUNetTrainer_100epochs__nnUNetPlans__2d"
assert trainer_base.exists(), f"Missing trainer directory: {trainer_base}"

for fold in [1, 2]:
    fold_dir = trainer_base / f"fold_{fold}"
    assert fold_dir.exists(), f"Missing fold directory: {fold_dir}"
    
    # Backup checkpoint_final.pth
    dst_fold_backup = out_dir / f"weights_fold_{fold}"
    dst_fold_backup.mkdir(parents=True, exist_ok=True)
    
    for f in ["checkpoint_final.pth", "checkpoint_best.pth", "summary.json", "progress.png"]:
        src_f = fold_dir / f
        if src_f.exists():
            shutil.copy2(src_f, dst_fold_backup / f)
            print(f"Saved Fold {fold} {f} ({src_f.stat().st_size / 1024 / 1024:.2f} MB)")
            
    val_dir = fold_dir / "validation"
    if val_dir.exists():
        dst_val = out_dir / f"validation_fold{fold}"
        shutil.copytree(val_dir, dst_val, dirs_exist_ok=True)
        print(f"Saved Fold {fold} validation masks: {len(list(dst_val.glob('*.png')))} masks")

print("All backups completed successfully.")
